# Fil rouge : boîte à outils

Ce notebook **fonctionne tel quel** : il entraîne un ResNet18 et mesure tout ce qu'il faut. À
vous de le modifier pour explorer la piste que vous avez choisie : <https://bonben.github.io/ensta-tp-deep-learning/04_fil_rouge/>.

Sur Colab : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et code du réseau

In [ ]:
import os
import tarfile
import urllib.request

SITE = "https://bonben.github.io/ensta-tp-deep-learning"
CIFAR_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/cifar-10-python.tar.gz"

if not os.path.exists("resnet.py"):
    urllib.request.urlretrieve(f"{SITE}/code/resnet.py", "resnet.py")
if not os.path.exists("data/cifar-10-batches-py"):
    urllib.request.urlretrieve(CIFAR_URL, "cifar.tar.gz")
    with tarfile.open("cifar.tar.gz") as tar:
        tar.extractall("data", filter="data")
    os.remove("cifar.tar.gz")
print("resnet.py et CIFAR-10 sont prêts")

## Les outils

- `count_params(model)` : nombre de paramètres appris ;
- `count_macs(model)` : multiplications-additions pour classer **une** image ;
- `fit(model, epochs, ...)` : entraîne et évalue. **Tous les réglages sont des arguments**,
  avec la recette de la baseline par défaut. À la fin, il affiche une ligne prête à recopier
  dans le tableau partagé.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
from torch.utils.flop_counter import FlopCounterMode
import matplotlib.pyplot as plt
from resnet import ResNet, BasicBlock

torch.multiprocessing.set_start_method("fork", force=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
NORMALIZE = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)]

# L'augmentation de la baseline (cours 2). Pour en changer, créez votre propre Compose.
transform_train = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip()] + NORMALIZE)
transform_val = v2.Compose(NORMALIZE)
val_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_macs(model):
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, 32, 32, device=next(model.parameters()).device))
    return counter.get_total_flops() // 2


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            correct += (model(inputs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


def fit(model, epochs, lr=0.1, weight_decay=5e-4, batch_size=128, label_smoothing=0.0,
        transform=transform_train, seed=0, name="essai"):
    """Entraîne avec SGD (momentum 0.9) et un scheduler cosinus. Renvoie l'historique."""
    torch.manual_seed(seed)
    train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        history["train_acc"].append(100 * correct / total)
        history["val_acc"].append(evaluate(model, val_loader))
        print(f"époque {epoch + 1}/{epochs} | train {history['train_acc'][-1]:.1f} % | "
              f"val {history['val_acc'][-1]:.1f} % | {time.time() - t0:.0f} s")
    print(f"\nÀ RECOPIER DANS LE TABLEAU : {name} | {count_params(model)} paramètres | "
          f"{count_macs(model)} MAC | {epochs} époques | graine {seed} | "
          f"val au dernier epoch {history['val_acc'][-1]:.2f} %")
    return history


def plot(*histories, labels=None):
    """Superpose les courbes de validation de plusieurs essais."""
    for i, h in enumerate(histories):
        plt.plot(h["val_acc"], label=labels[i] if labels else f"essai {i + 1}")
    plt.xlabel("époque"); plt.ylabel("précision de validation (%)")
    plt.grid(alpha=0.3); plt.legend(); plt.show()

## Un premier essai

Le point de départ : votre baseline, un ResNet18. Changez le modèle et les arguments de `fit`
selon votre piste. **Affichez toujours paramètres et MAC avant d'entraîner** : c'est gratuit,
et ça évite d'attendre 10 minutes pour un réseau trop gros.

In [ ]:
model = ResNet(BasicBlock, [2, 2, 2, 2], width=64)
print(f"{count_params(model):,} paramètres, {count_macs(model.to(device)):,} MAC")

In [ ]:
h_base = fit(model, epochs=15, name="ResNet18 baseline")

## Vos essais

Une cellule par essai, avec un nom qui dit ce que vous avez changé. Comparez avec `plot` :

```python
plot(h_base, h_essai1, labels=["baseline", "essai 1"])
```

In [ ]:
# model = ...
# h_essai1 = fit(model, epochs=15, name="...")